# Assignment 2 DSC 102 FA25

## Introduction

In this assignment we will conduct data engineering for the Amazon dataset. It is divided into 2 parts. The extracted features in Part 1 will be used for the Part 2 of assignment, where you train a model (or models) to predict user ratings for a product.

We will be using Apache Spark for this assignment. The default Spark API will be DataFrame, as it is now the recommended choice over the RDD API. That being said, please feel free to switch back to the RDD API if you see it as a better fit for the task. We provide you an option to request RDD format to start with. Also you can switch between DataFrame and RDD in your solution. 

Another newer API is Koalas, which is also avaliable. However, it has constraints and is not applicable to most tasks. Refer to the PA statement for detail.

### Set the following parameters

In [1]:
PID = 'A17887951' # your pid, for instance: 'a43223333'
INPUT_FORMAT = 'dataframe' # choose a format of your input data, valid options: 'dataframe', 'rdd', 'koalas'

In [2]:
# Boiler plates, do NOT modify
%load_ext autoreload
%autoreload 2
import os
import getpass
from pyspark.sql import SparkSession
from utilities import SEED
from utilities import PA2Test
from utilities import PA2Data
from utilities import data_cat
from pa2_main import PA2Executor
import time
if INPUT_FORMAT == 'dataframe':
    import pyspark.ml as M
    import pyspark.sql.functions as F
    import pyspark.sql.types as T
if INPUT_FORMAT == 'koalas':
    import databricks.koalas as ks
elif INPUT_FORMAT == 'rdd':
    import pyspark.mllib as M
    from pyspark.mllib.feature import Word2Vec
    from pyspark.mllib.linalg import Vectors
    from pyspark.mllib.linalg.distributed import RowMatrix

os.environ['PYSPARK_SUBMIT_ARGS'] = '--py-files utilities.py,assignment2.py \
--deploy-mode client \
pyspark-shell'

class args:
    review_filename = data_cat.review_filename
    product_filename = data_cat.product_filename
    product_processed_filename = data_cat.product_processed_filename
    ml_features_train_filename = data_cat.ml_features_train_filename
    ml_features_test_filename = data_cat.ml_features_test_filename
    output_root = '/home/{}/{}-pa2/test_results'.format(getpass.getuser(), PID)
    test_results_root = data_cat.test_results_root
    pid = PID

pa2 = PA2Executor(args, input_format=INPUT_FORMAT)
data_io = pa2.data_io
data_dict = pa2.data_dict
begin = time.time()


26/05/29 17:39:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/bitnami/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Loading datasets ...

Done


In [ ]:
# Import your own dependencies



#-----------------------------

# Part 1: Feature Engineering

In [3]:
# Bring the part_1 datasets to memory and de-cache part_2 datasets. 
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_1')

# Task0: warm up 
This task is provided for you to get familiar with Spark API. We will use the dataframe API to demonstrate. Solution is given to you and this task won't be graded.

Refer to https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html for API guide.

The task is to implement the function below. Given the ```product_data``` table:
1. Take and print five rows.

1. Select only the ```asin``` column, then print five rows of it.

1. Select the row where ```asin = B00I8KEOTM``` and print it.

1. Count the total number of rows.

1. Calculate the mean ```price```.

1. You need to conduct the above operations, then extract some statistics out of the generated columns. You need to put the statistics in a python dictionary named ```res```. The description and schema of it are as follows:
    ```
    res
     | -- count_total: int -- count of total rows of the entire table after your operations
     | -- mean_price: float -- mean value of column price
    ```

In [4]:
def task_0(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    product_data.show(5)
    product_data[['asin']].show(5)
    product_data.where(F.col('asin') == 'B00I8KEOTM').show()
    count_rows = product_data.count()
    mean_price = product_data.select(F.avg(F.col('price'))).head()[0]
    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmatically. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {'count_total': None, 'mean_price': None}
    
    # Modify res:

    res['count_total'] = count_rows
    res['mean_price'] = mean_price

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    return res
    # -------------------------------------------------------------------------

In [5]:
if INPUT_FORMAT == 'dataframe':
    res = task_0(data_io, data_dict['product'])
    pa2.tests.test(res, 'task_0')

+----------+--------------------+--------------------+--------------------+-----+--------------------+
|      asin|           salesRank|          categories|               title|price|             related|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
|B00I8HVV6E|{Home &amp; Kitch...|[[Home & Kitchen,...|Intelligent Desig...|27.99|{also_viewed -> [...|
|B00I8KEOTM|                null|[[Apps for Androi...|                null| null|{also_viewed -> [...|
|B00I8KCW4G|{Clothing -> 2233...|[[Clothing, Shoes...|eShakti Women's P...|41.95|{also_viewed -> [...|
|B00I8JKCQW|{Clothing -> 1405...|[[Clothing, Shoes...|Lady Slimming Mid...| null|{also_viewed -> [...|
|B00I8JKI8E|{Home &amp; Kitch...|[[Clothing, Shoes...|3 Tier Bangle Bra...|24.99|{also_viewed -> [...|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
only showing top 5 rows

+----------+
|      asin|
+----------+
|B00I8HVV

+----------+---------+--------------------+-----+-----+--------------------+
|      asin|salesRank|          categories|title|price|             related|
+----------+---------+--------------------+-----+-----+--------------------+
|B00I8KEOTM|     null|[[Apps for Androi...| null| null|{also_viewed -> [...|
+----------+---------+--------------------+-----+-----+--------------------+



tests for task_0 --------------------------------------------------------------
2/2 passed
-------------------------------------------------------------------------------


# Task1

In [26]:
# %load -s task_1 assignment2.py
def task_1(data_io, review_data, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    ratings = review_data.groupBy("asin").agg(
        F.avg("overall").alias("meanRating"),
        F.count("overall").alias("countRating")
    )
    
    transformed = product_data.join(ratings, on="asin", how="left")
    
    stats = transformed.agg(
        F.count("*").alias("count_total"),

        F.mean("meanRating").alias("mean_meanRating"),
        F.variance("meanRating").alias("variance_meanRating"),
        F.sum(F.col("meanRating").isNull().cast("int")).alias("numNulls_meanRating"),

        F.mean("countRating").alias("mean_countRating"),
        F.variance("countRating").alias("variance_countRating"),
        F.sum(F.col("countRating").isNull().cast("int")).alias("numNulls_countRating")
    ).collect()[0]


    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmaticly. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {
        'count_total': None,
        'mean_meanRating': None,
        'variance_meanRating': None,
        'numNulls_meanRating': None,
        'mean_countRating': None,
        'variance_countRating': None,
        'numNulls_countRating': None
    }
    # Modify res:
    res["count_total"] = int(stats["count_total"])
    res["mean_meanRating"] = float(stats["mean_meanRating"])
    res["variance_meanRating"] = float(stats["variance_meanRating"])
    res["numNulls_meanRating"] = int(stats["numNulls_meanRating"])
    res["mean_countRating"] = float(stats["mean_countRating"])
    res["variance_countRating"] = float(stats["variance_countRating"])
    res["numNulls_countRating"] = int(stats["numNulls_countRating"])


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_1')
    return res
    # -------------------------------------------------------------------------


In [27]:
res = task_1(data_io, data_dict['review'], data_dict['product'])
pa2.tests.test(res, 'task_1')

tests for task_1 --------------------------------------------------------------
Test 1/7 : count_total ... Pass
Test 2/7 : mean_countRating ... Pass
Test 3/7 : mean_meanRating ... Pass
Test 4/7 : numNulls_countRating ... Pass
Test 5/7 : numNulls_meanRating ... Pass
Test 6/7 : variance_countRating ... Pass
Test 7/7 : variance_meanRating ... Pass
7/7 passed
-------------------------------------------------------------------------------


True


# Task 2

In [22]:
# %load -s task_2 assignment2.py
def task_2(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    salesRank_column = 'salesRank'
    categories_column = 'categories'
    asin_column = 'asin'
    # Outputs:
    category_column = 'category'
    bestSalesCategory_column = 'bestSalesCategory'
    bestSalesRank_column = 'bestSalesRank'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    raw_category = F.col("categories").getItem(0).getItem(0)

    product_with_category = product_data.withColumn(
        "category",
        F.when(
            F.col("categories").isNull()
            | (F.size(F.col("categories")) == 0)
            | F.col("categories").getItem(0).isNull()
            | (F.size(F.col("categories").getItem(0)) == 0)
            | raw_category.isNull()
            | (raw_category == ""),
            F.lit(None)
        ).otherwise(raw_category)
    )
    
    product_with_all = product_with_category.withColumn(
        "bestSalesCategory",
        F.when(
            F.col("salesRank").isNull() | (F.size(F.map_keys("salesRank")) == 0),
            F.lit(None)
        ).otherwise(F.map_keys("salesRank")[0])
    )

    product_with_all = product_with_all.withColumn(
        "bestSalesRank",
        F.when(
            F.col("salesRank").isNull() | (F.size(F.map_values("salesRank")) == 0),
            F.lit(None)
        ).otherwise(F.map_values("salesRank")[0])
    )

    stats = product_with_all.agg(
        F.count("*").alias("count_total"),

        F.mean("bestSalesRank").alias("mean_bestSalesRank"),
        F.variance("bestSalesRank").alias("variance_bestSalesRank"),

        F.sum(F.col("category").isNull().cast("int")).alias("numNulls_category"),
        F.countDistinct("category").alias("countDistinct_category"),

        F.sum(F.col("bestSalesCategory").isNull().cast("int")).alias("numNulls_bestSalesCategory"),
        F.countDistinct("bestSalesCategory").alias("countDistinct_bestSalesCategory")
    ).collect()[0]



    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'mean_bestSalesRank': None,
        'variance_bestSalesRank': None,
        'numNulls_category': None,
        'countDistinct_category': None,
        'numNulls_bestSalesCategory': None,
        'countDistinct_bestSalesCategory': None
    }
    # Modify res:
    res["count_total"] = int(stats["count_total"])
    res["mean_bestSalesRank"] = float(stats["mean_bestSalesRank"])
    res["variance_bestSalesRank"] = float(stats["variance_bestSalesRank"])
    res["numNulls_category"] = int(stats["numNulls_category"])
    res["countDistinct_category"] = int(stats["countDistinct_category"])
    res["numNulls_bestSalesCategory"] = int(stats["numNulls_bestSalesCategory"])
    res["countDistinct_bestSalesCategory"] = int(stats["countDistinct_bestSalesCategory"])
    

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_2')
    return res
    # -------------------------------------------------------------------------


In [23]:
res = task_2(data_io, data_dict['product'])
pa2.tests.test(res, 'task_2')

tests for task_2 --------------------------------------------------------------
Test 1/7 : countDistinct_bestSalesCategory ... Pass
Test 2/7 : countDistinct_category ... Pass
Test 3/7 : count_total ... Pass
Test 4/7 : mean_bestSalesRank ... Pass
Test 5/7 : numNulls_bestSalesCategory ... Pass
Test 6/7 : numNulls_category ... Pass
Test 7/7 : variance_bestSalesRank ... Pass
7/7 passed
-------------------------------------------------------------------------------


True

# Task 3





In [28]:
# %load -s task_3 assignment2.py
def task_3(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    price_column = 'price'
    attribute = 'also_viewed'
    related_column = 'related'
    # Outputs:
    meanPriceAlsoViewed_column = 'meanPriceAlsoViewed'
    countAlsoViewed_column = 'countAlsoViewed'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    also_viewed = F.col("related").getItem("also_viewed")
    
    base = product_data.withColumn(
        "countAlsoViewed",
        F.when(
            F.col("related").isNull()
            | also_viewed.isNull()
            | (F.size(also_viewed) == 0),
            F.lit(None)
        ).otherwise(F.size(also_viewed))
    )
    
    price_lookup = product_data.select(
        F.col("asin").alias("viewed_asin"),
        F.col("price").alias("viewed_price")
    )
    
    exploded = base.select(
        F.col("asin").alias("source_asin"),
        F.explode_outer(also_viewed).alias("viewed_asin")
    )

    mean_prices = exploded.join(
        price_lookup,
        on="viewed_asin",
        how="left"
    ).groupBy("source_asin").agg(
        F.mean("viewed_price").alias("meanPriceAlsoViewed")
    )
    
    transformed = base.join(
        mean_prices,
        base.asin == mean_prices.source_asin,
        how="left"
    ).drop("source_asin")
    
    stats = transformed.agg(
        F.count("*").alias("count_total"),

        F.mean("meanPriceAlsoViewed").alias("mean_meanPriceAlsoViewed"),
        F.variance("meanPriceAlsoViewed").alias("variance_meanPriceAlsoViewed"),
        F.sum(F.col("meanPriceAlsoViewed").isNull().cast("int")).alias("numNulls_meanPriceAlsoViewed"),

        F.mean("countAlsoViewed").alias("mean_countAlsoViewed"),
        F.variance("countAlsoViewed").alias("variance_countAlsoViewed"),
        F.sum(F.col("countAlsoViewed").isNull().cast("int")).alias("numNulls_countAlsoViewed")
    ).collect()[0]


    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'mean_meanPriceAlsoViewed': None,
        'variance_meanPriceAlsoViewed': None,
        'numNulls_meanPriceAlsoViewed': None,
        'mean_countAlsoViewed': None,
        'variance_countAlsoViewed': None,
        'numNulls_countAlsoViewed': None
    }
    # Modify res:
    res["count_total"] = int(stats["count_total"])

    res["mean_meanPriceAlsoViewed"] = float(stats["mean_meanPriceAlsoViewed"])
    res["variance_meanPriceAlsoViewed"] = float(stats["variance_meanPriceAlsoViewed"])
    res["numNulls_meanPriceAlsoViewed"] = int(stats["numNulls_meanPriceAlsoViewed"])

    res["mean_countAlsoViewed"] = float(stats["mean_countAlsoViewed"])
    res["variance_countAlsoViewed"] = float(stats["variance_countAlsoViewed"])
    res["numNulls_countAlsoViewed"] = int(stats["numNulls_countAlsoViewed"])


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_3')
    return res
    # -------------------------------------------------------------------------


In [29]:
res = task_3(data_io, data_dict['product'])
pa2.tests.test(res, 'task_3')

tests for task_3 --------------------------------------------------------------
Test 1/7 : count_total ... Pass
Test 2/7 : mean_countAlsoViewed ... Pass
Test 3/7 : mean_meanPriceAlsoViewed ... Pass
Test 4/7 : numNulls_countAlsoViewed ... Pass
Test 5/7 : numNulls_meanPriceAlsoViewed ... Pass
Test 6/7 : variance_countAlsoViewed ... Pass
Test 7/7 : variance_meanPriceAlsoViewed ... Pass
7/7 passed
-------------------------------------------------------------------------------


True

# Task 4

In [32]:
# %load -s task_4 assignment2.py
def task_4(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    price_column = 'price'
    title_column = 'title'
    # Outputs:
    meanImputedPrice_column = 'meanImputedPrice'
    medianImputedPrice_column = 'medianImputedPrice'
    unknownImputedTitle_column = 'unknownImputedTitle'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    product_float = product_data.withColumn(
        "price_float",
        F.col("price").cast("float")
    )

    mean_price = product_float.agg(
        F.mean("price_float").alias("mean_price")
    ).collect()[0]["mean_price"]

    median_price = product_float.approxQuantile(
        "price_float",
        [0.5],
        0.001
    )[0]

    transformed = product_float.withColumn(
        "meanImputedPrice",
        F.when(
            F.col("price_float").isNull(),
            F.lit(float(mean_price))
        ).otherwise(F.col("price_float"))
    ).withColumn(
        "medianImputedPrice",
        F.when(
            F.col("price_float").isNull(),
            F.lit(float(median_price))
        ).otherwise(F.col("price_float"))
    ).withColumn(
        "unknownImputedTitle",
        F.when(
            F.col("title").isNull() | (F.col("title") == ""),
            F.lit("unknown")
        ).otherwise(F.col("title"))
    )

    stats = transformed.agg(
        F.count("*").alias("count_total"),

        F.mean("meanImputedPrice").alias("mean_meanImputedPrice"),
        F.variance("meanImputedPrice").alias("variance_meanImputedPrice"),
        F.sum(F.col("meanImputedPrice").isNull().cast("int")).alias("numNulls_meanImputedPrice"),

        F.mean("medianImputedPrice").alias("mean_medianImputedPrice"),
        F.variance("medianImputedPrice").alias("variance_medianImputedPrice"),
        F.sum(F.col("medianImputedPrice").isNull().cast("int")).alias("numNulls_medianImputedPrice"),

        F.sum((F.col("unknownImputedTitle") == "unknown").cast("int")).alias("numUnknowns_unknownImputedTitle")
    ).collect()[0]


    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'mean_meanImputedPrice': None,
        'variance_meanImputedPrice': None,
        'numNulls_meanImputedPrice': None,
        'mean_medianImputedPrice': None,
        'variance_medianImputedPrice': None,
        'numNulls_medianImputedPrice': None,
        'numUnknowns_unknownImputedTitle': None
    }
    # Modify res:
    res["count_total"] = int(stats["count_total"])
    res["mean_meanImputedPrice"] = float(stats["mean_meanImputedPrice"])
    res["variance_meanImputedPrice"] = float(stats["variance_meanImputedPrice"])
    res["numNulls_meanImputedPrice"] = int(stats["numNulls_meanImputedPrice"])
    res["mean_medianImputedPrice"] = float(stats["mean_medianImputedPrice"])
    res["variance_medianImputedPrice"] = float(stats["variance_medianImputedPrice"])
    res["numNulls_medianImputedPrice"] = int(stats["numNulls_medianImputedPrice"])
    res["numUnknowns_unknownImputedTitle"] = float(stats["numUnknowns_unknownImputedTitle"])


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_4')
    return res
    # -------------------------------------------------------------------------


In [33]:
res = task_4(data_io, data_dict['product'])
pa2.tests.test(res, 'task_4')

tests for task_4 --------------------------------------------------------------
Test 1/8 : count_total ... Pass
Test 2/8 : mean_meanImputedPrice ... Pass
Test 3/8 : mean_medianImputedPrice ... Pass
Test 4/8 : numNulls_meanImputedPrice ... Pass
Test 5/8 : numNulls_medianImputedPrice ... Pass
Test 6/8 : numUnknowns_unknownImputedTitle ... Pass
Test 7/8 : variance_meanImputedPrice ... Pass
Test 8/8 : variance_medianImputedPrice ... Pass
8/8 passed
-------------------------------------------------------------------------------


True

# Task 5

In [ ]:
# %load -s task_5 assignment2.py
def task_5(data_io, product_processed_data, word_0, word_1, word_2):
    # -----------------------------Column names--------------------------------
    # Inputs:
    title_column = 'title'
    # Outputs:
    titleArray_column = 'titleArray'
    titleVector_column = 'titleVector'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------





    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'size_vocabulary': None,
        'word_0_synonyms': [(None, None), ],
        'word_1_synonyms': [(None, None), ],
        'word_2_synonyms': [(None, None), ]
    }
    # Modify res:
    res['count_total'] = product_processed_data_output.count()
    res['size_vocabulary'] = model.getVectors().count()
    for name, word in zip(
        ['word_0_synonyms', 'word_1_synonyms', 'word_2_synonyms'],
        [word_0, word_1, word_2]
    ):
        res[name] = model.findSynonymsArray(word, 10)
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_5')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_5(data_io, data_dict['product_processed'], 'piano', 'rice', 'laptop')
pa2.tests.test(res, 'task_5')

# Task 6

In [ ]:
# %load -s task_6 assignment2.py
def task_6(data_io, product_processed_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    category_column = 'category'
    # Outputs:
    categoryIndex_column = 'categoryIndex'
    categoryOneHot_column = 'categoryOneHot'
    categoryPCA_column = 'categoryPCA'
    # -------------------------------------------------------------------------    

    # ---------------------- Your implementation begins------------------------





    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'meanVector_categoryOneHot': [None, ],
        'meanVector_categoryPCA': [None, ]
    }
    # Modify res:




    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_6')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_6(data_io, data_dict['product_processed'])
pa2.tests.test(res, 'task_6')

In [ ]:
print ("End to end time: {}".format(time.time()-begin))

# Part 2: Model Selection

In [ ]:
# Bring the part_2 datasets to memory and de-cache part_1 datasets.
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_2')

# Task 7

In [ ]:
def task_7(data_io, train_data, test_data):
    
    # ---------------------- Your implementation begins------------------------
    
    
    
    
    
    # -------------------------------------------------------------------------
    
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': None
    }
    # Modify res:


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_7')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_7(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_7')

# Task 8

In [ ]:
def task_8(data_io, train_data, test_data):
    
    # ---------------------- Your implementation begins------------------------
    
    
    
    
    
    # -------------------------------------------------------------------------
    
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': None,
        'valid_rmse_depth_5': None,
        'valid_rmse_depth_7': None,
        'valid_rmse_depth_9': None,
        'valid_rmse_depth_12': None,
    }
    # Modify res:


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_8')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_8(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_8')

In [ ]:
print ("End to end time: {}".format(time.time()-begin))